# Week 06 — Function 02

## Table of contents

1. [Overview](#overview)
2. [Objectives](#objectives)
3. [Evidence provenance](#evidence-provenance)
4. [Environment and setup](#environment-setup)
5. [Data validation](#data-validation)
6. [Descriptive EDA](#descriptive-eda)
7. [Visual EDA](#visual-eda)
8. [Model and acquisition](#model-acquisition)
9. [Week 6 proposal](#week-6-proposal)
10. [Reproducibility checks](#reproducibility-checks)
11. [Conclusions and next steps](#conclusions-next-steps)

<a id="overview"></a>
## 1. Overview

This focused review mirrors the canonical Week 6 methodology for Function 02, with Weeks 1–5 observed and Week 6 proposed only.

<a id="objectives"></a>
## 2. Objectives

Validate the 2-dimensional evidence, assess the latest returned point, and reproduce the recorded GP-UCB proposal without look-ahead.

<a id="evidence-provenance"></a>
## 3. Evidence provenance

Starter arrays come from `Week_01/Function_nn/03_Data`; exact returned pairs come from `Results/query_output_ledger.csv`. The Week 6 return is excluded because it was unknown when the proposal was selected.

<a id="environment-setup"></a>
## 4. Environment and setup

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT=Path.cwd().resolve()
for candidate in (ROOT,*ROOT.parents):
    if (candidate/'Week_06'/'Function_02').is_dir(): ROOT=candidate; break
else: raise FileNotFoundError('Could not locate repository root')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from Code.historical_function_review import analyse_historical_function

<a id="data-validation"></a>
## 5. Data validation

In [2]:
observations, summary, proposal, diagnostic_figure = analyse_historical_function(6, 2, ROOT)
input_columns=[f'x{i}' for i in range(1,3)]
inputs=observations[input_columns].to_numpy(float)
outputs=observations['objective'].to_numpy(float)
assert inputs.shape==(15,2) and outputs.shape==(15,)
assert np.isfinite(inputs).all() and np.isfinite(outputs).all()
assert np.all((inputs>=0)&(inputs<=1))
observations

,query,evidence,x1,x2,objective
0,1,starter,0.665800,0.123969,0.538996
1,2,starter,0.877791,0.778628,0.420586
2,3,starter,0.142699,0.349005,-0.065624
3,4,starter,0.845275,0.711120,0.293993
4,5,starter,0.454647,0.290455,0.214965
5,6,starter,0.577713,0.771973,0.023106
6,7,starter,0.438166,0.685018,0.244619
7,8,starter,0.341750,0.028698,0.038749
8,9,starter,0.338648,0.213867,-0.013858
9,10,starter,0.702637,0.926564,0.611205


<a id="descriptive-eda"></a>
## 6. Descriptive EDA

All comparisons are descriptive and within-function; no causal, global-optimum, or cross-function ranking claim is made.

In [3]:
pd.Series({k:v for k,v in summary.items() if k!='proposal'}, name='verified evidence')

week                                                                             6
function                                                                         2
dimensions                                                                       2
starter_observations                                                            10
recorded_pairs                                                                   5
total_verified_observations                                                     15
best_query                                                                      10
best_input                                [0.7026365569244406, 0.9265641975455574]
best_output                                                               0.611205
latest_verified_query                                                           15
latest_verified_input                                         [0.555332, 0.360931]
latest_verified_output                                                    0.220781
late

<a id="visual-eda"></a>
## 7. Visual EDA

Orange markers are returned Weeks 1–5 observations; the star is the verified incumbent. The Week 6 proposal is deliberately absent.

In [4]:
display(diagnostic_figure)
plt.close(diagnostic_figure)

<Figure size 1200x450 with 3 Axes>

<a id="model-acquisition"></a>
## 8. Model and acquisition

Method: **GP-UCB**. This adaptive policy is a heuristic chosen from the evidence available at the decision boundary; it is not a statistically controlled acquisition comparison.

<a id="week-6-proposal"></a>
## 9. Week 6 proposal

Proposed only: `[0.473151, 0.950706]`. Decision record: Chosen using evidence through Week 5, before the Week 6 return.

<a id="reproducibility-checks"></a>
## 10. Reproducibility checks

In [5]:
candidate=np.asarray(proposal['query'],dtype=float)
assert proposal['status']=='proposed_only'
assert candidate.shape==(2,) and np.all((candidate>=0)&(candidate<=0.999999))
duplicate=bool(np.any(np.all(np.isclose(inputs,candidate,rtol=0,atol=5e-7),axis=1)))
assert duplicate==proposal['duplicates_observed_evidence']
assert summary['recorded_pairs']==5
portal='-'.join(f'{value:.6f}' for value in candidate)
assert all(len(part.split('.')[-1])==6 for part in portal.split('-'))
print('Function 02 Week 6 checks passed:', portal, 'duplicate:', duplicate)

Function 02 Week 6 checks passed: 0.473151-0.950706 duplicate: False


<a id="conclusions-next-steps"></a>
## 11. Conclusions and next steps

The evidence boundary is locked at 15 verified observations. The Week 6 proposal remains unobserved until its authoritative return is appended at the next checkpoint.